**IMPORTS**

In [1]:
import os
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DeiTFeatureExtractor, ViTModel
from PIL import Image
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**CONFIG**

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_unixcoder_nn/"
BATCH_SIZE = 8

UNIXCODER_CKPT = "../checkpoints/unixcoder_only/checkpoint-2322/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../Text_Files/Train"
IMAGE_DIR = "../snapshots/Train"
TEST_BASE = "../snapshots"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

**LOAD MODELS**

In [3]:
from transformers import ViTForImageClassification, AutoImageProcessor

print("Loading fine-tuned UnixCoder...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")
text_model = AutoModel.from_pretrained(UNIXCODER_CKPT).to(DEVICE)
text_model.eval()

Loading fine-tuned UnixCoder...


Some weights of RobertaModel were not initialized from the model checkpoint at ../checkpoints/unixcoder_only/checkpoint-2322/ and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(51416, 768, padding_idx=1)
    (position_embeddings): Embedding(1026, 768, padding_idx=1)
    (token_type_embeddings): Embedding(10, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (

In [4]:
print("Loading fine-tuned DeiT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit# This gives you the pure ViTModel
vit_model.to(DEVICE)
vit_model.eval()

Loading fine-tuned DeiT model...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

**DATASET**

In [5]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r") as f:
            text = f.read()
        encoding = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label

**DATA LOADING**

In [6]:
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)  # No shuffle for feature extraction

**FEATURE EXTRACTION FOR RANDOM FOREST**

In [8]:
class FeatureExtractor(nn.Module):
    def __init__(self, text_model, vit_model):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # freeze backbone models
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask, image_tensor):
        # Text embedding: CLS token
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = text_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Image embedding: CLS token
        image_outputs = self.vit_model(**{k: v for k, v in image_tensor.items()})
        image_cls = image_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Concatenate features
        fused = torch.cat([text_cls, image_cls], dim=1)
        return fused

In [9]:
feature_extractor = FeatureExtractor(text_model, vit_model).to(DEVICE)
feature_extractor.eval()

print("Extracting features for Random Forest training...")

all_features = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(train_loader):
        input_ids, attention_mask, image_tensor, labels = batch
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        features = feature_extractor(input_ids, attention_mask, image_tensor)
        
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Concatenate all batches
X_train = np.concatenate(all_features, axis=0)
y_train = np.concatenate(all_labels, axis=0)

print(f"Features shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")

Extracting features for Random Forest training...


100%|██████████| 774/774 [01:22<00:00,  9.41it/s]

Features shape: (6190, 1536)
Labels shape: (6190,)


**TRAIN RANDOM FOREST CLASSIFIER**

In [10]:
print("Training Random Forest Classifier...")

# Initialize and train Random Forest
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)
print("Random Forest training completed!")

# Check training accuracy
train_preds = rf_classifier.predict(X_train)
train_accuracy = accuracy_score(y_train, train_preds)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

Training Random Forest Classifier...
Random Forest training completed!
Training Accuracy: 99.66%


**SAVE RANDOM FOREST MODEL**

In [11]:
import joblib

# Save the trained Random Forest model
rf_model_path = os.path.join(OUTPUT_DIR, "random_forest_model.pkl")
joblib.dump(rf_classifier, rf_model_path)
print(f"Random Forest model saved to: {rf_model_path}")

Random Forest model saved to: ../checkpoints/early_fusion_unixcoder_rf/random_forest_model.pkl


**TESTING FUNCTION**

In [12]:
def test_on_dataset_rf(text_dir, image_dir, tokenizer, image_processor, feature_extractor, rf_model, batch_size=4):
    """🧪 Test the fusion model with Random Forest on a single dataset"""

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    feature_extractor.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Features"):
            input_ids, attention_mask, image_tensor, labels = batch

            # Move text to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            # Move image tensors to device
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Extract features
            features = feature_extractor(input_ids=input_ids, attention_mask=attention_mask, image_tensor=image_tensor)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Concatenate all batches
    X_test = np.concatenate(all_features, axis=0)
    y_test = np.concatenate(all_labels, axis=0)

    # Predict with Random Forest
    preds = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    
    print(f"✅ Dataset Accuracy: {accuracy*100:.2f}% 🎉")
    return accuracy

**TESTING ON ALL DATASETS**

In [13]:
print("=== Testing Random Forest Fusion Model ===")

accuracies = []
for i in range(10):
    print(f"Test_{i}")
    accuracy = test_on_dataset_rf(
        f"../Text_Files/Test_{i}", 
        f"../snapshots/Test_{i}", 
        tokenizer, 
        image_processor, 
        feature_extractor, 
        rf_classifier, 
        batch_size=4
    )
    accuracies.append(accuracy)

print(f"\n=== Final Results ===")
print(f"Average Accuracy: {np.mean(accuracies) * 100:.2f}%")
print(f"Standard Deviation: {np.std(accuracies) * 100:.2f}%")

=== Testing Random Forest Fusion Model ===
Test_0


Extracting Features: 100%|██████████| 251/251 [00:13<00:00, 17.99it/s]


✅ Dataset Accuracy: 87.52% 🎉
Test_1


Extracting Features: 100%|██████████| 251/251 [00:13<00:00, 18.09it/s]


✅ Dataset Accuracy: 81.04% 🎉
Test_2


...

✅ Dataset Accuracy: 72.36% 🎉
Test_8


Extracting Features: 100%|██████████| 251/251 [00:13<00:00, 18.15it/s]


✅ Dataset Accuracy: 78.64% 🎉
Test_9


Extracting Features: 100%|██████████| 251/251 [00:13<00:00, 18.27it/s]

✅ Dataset Accuracy: 72.36% 🎉

=== Final Results ===
Average Accuracy: 79.76%
Standard Deviation: 5.01%
